# Limpieza de movimientos de repuestos para mantenimiento portuario

**Proyecto:** gestión de mantenimiento de equipos portuarios  
**Objetivo:** limpiar, clasificar y consolidar los movimientos históricos de repuestos para análisis en Power BI.

El notebook procesa automáticamente los archivos `REPUESTOS *.xlsx`, los cruza con la tabla maestra y genera informes mensuales y un consolidado. No reutiliza archivos mensuales de ejecuciones anteriores.


## 1. Librerías, rutas y nombres de los informes


In [ ]:
from pathlib import Path
import re
import unicodedata

import pandas as pd
from IPython.display import display


def localizar_carpeta_etl():
    """Localiza ETL sin depender de la carpeta desde la que se inicia Jupyter."""
    actual = Path.cwd().resolve()
    candidatos = []
    for base in (actual, *actual.parents):
        candidatos.extend((base, base / "ETL"))

    for carpeta in candidatos:
        if (carpeta / "Data").is_dir() and (carpeta / "src").is_dir():
            return carpeta

    rutas = "\n- ".join(str(ruta) for ruta in candidatos)
    raise FileNotFoundError("No se encontró la carpeta ETL. Rutas revisadas:\n- " + rutas)


ETL_DIR = localizar_carpeta_etl()
DATA_DIR = ETL_DIR / "Data"
RESULTADOS_DIR = ETL_DIR / "Resultado"
CARPETA_SALIDA_MENSUAL = ETL_DIR / "Informes Mantenimiento Portuario"

NOMBRE_MAESTRO = "Informe_Maestro_Repuestos_Mantenimiento_Equipos_Portuarios.xlsx"
NOMBRE_CONSOLIDADO = "Informe_Consolidado_Movimientos_Repuestos_Mantenimiento_Portuario.xlsx"
NOMBRE_NO_CLASIFICADOS = "Informe_Repuestos_No_Clasificados_Mantenimiento_Portuario.xlsx"
PREFIJO_MENSUAL = "Informe_Movimientos_Repuestos_Mantenimiento_Portuario"

RESULTADOS_DIR.mkdir(parents=True, exist_ok=True)
CARPETA_SALIDA_MENSUAL.mkdir(parents=True, exist_ok=True)

archivos_origen = sorted(DATA_DIR.glob("REPUESTOS 20*.xlsx"))
if not archivos_origen:
    raise FileNotFoundError(f"No se encontraron archivos 'REPUESTOS 20*.xlsx' en {DATA_DIR}")

# Se prioriza el nombre nuevo y se conserva compatibilidad con el archivo existente.
candidatos_maestro = [
    DATA_DIR / NOMBRE_MAESTRO,
    DATA_DIR / "Tabla_Maestra_Repuestos.xlsx",
    DATA_DIR / "Tabla_Maestra_Repuesto.xlsx",
]
ruta_tabla_maestra = next((ruta for ruta in candidatos_maestro if ruta.is_file()), None)
if ruta_tabla_maestra is None:
    raise FileNotFoundError(
        "No se encontró la tabla maestra de repuestos. Ejecuta primero "
        "'ExtraccionTablaMaestra.ipynb'."
    )

ruta_salida_powerbi = RESULTADOS_DIR / NOMBRE_CONSOLIDADO
ruta_no_clasificados = DATA_DIR / NOMBRE_NO_CLASIFICADOS

print("Archivos origen:")
for archivo in archivos_origen:
    print(f"- {archivo.name}")
print(f"Tabla maestra: {ruta_tabla_maestra.name}")


## 2. Funciones de normalización y detección

La hoja, el encabezado y la columna principal se detectan por contenido. Las filas que no contienen un código entre corchetes se descartan, evitando eliminaciones por posiciones fijas.


In [ ]:
MESES = (
    "ENERO|FEBRERO|MARZO|ABRIL|MAYO|JUNIO|JULIO|AGOSTO|"
    "SEPTIEMBRE|SETIEMBRE|OCTUBRE|NOVIEMBRE|DICIEMBRE"
)


def normalizar_texto(valor):
    if pd.isna(valor):
        return ""
    texto = unicodedata.normalize("NFKD", str(valor))
    texto = "".join(caracter for caracter in texto if not unicodedata.combining(caracter))
    return re.sub(r"\s+", " ", texto).strip().upper()


def normalizar_codigo(serie):
    codigos = (
        serie.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )
    numericos = codigos.str.fullmatch(r"\d+", na=False)
    codigos.loc[numericos] = codigos.loc[numericos].str.zfill(4)
    return codigos


def es_periodo(valor):
    texto = normalizar_texto(valor)
    return bool(
        re.search(rf"\b(?:{MESES})\b.*\b(?:19|20)\d{{2}}\b", texto)
        or re.fullmatch(r"(?:19|20)\d{2}[-_/](?:0?[1-9]|1[0-2])", texto)
    )


def detectar_estructura_movimientos(ruta, max_filas=20):
    libro = pd.ExcelFile(ruta)
    mejor = None

    for hoja in libro.sheet_names:
        muestra = pd.read_excel(ruta, sheet_name=hoja, header=None, nrows=max_filas)
        for fila, valores in muestra.iterrows():
            cantidad_periodos = sum(es_periodo(valor) for valor in valores)
            if cantidad_periodos and (mejor is None or cantidad_periodos > mejor[0]):
                mejor = (cantidad_periodos, hoja, fila)

    if mejor is None:
        raise ValueError(
            f"No se detectó una fila con periodos mensuales en '{ruta.name}'."
        )
    return mejor[1], mejor[2]


def limpiar_nombre_archivo(texto):
    texto = normalizar_texto(texto)
    texto = re.sub(r"[^A-Z0-9]+", "_", texto)
    return re.sub(r"_+", "_", texto).strip("_")


## 3. Carga de la tabla maestra de repuestos


In [ ]:
def cargar_tabla_maestra(ruta):
    libro = pd.ExcelFile(ruta)
    for hoja in libro.sheet_names:
        df = pd.read_excel(ruta, sheet_name=hoja)
        df.columns = [normalizar_texto(columna) for columna in df.columns]

        renombres = {}
        if "CODIGO_PRODUCTO" in df.columns:
            renombres["CODIGO_PRODUCTO"] = "CODIGO_REPUESTO"
        if "NOMBRE_PRODUCTO" in df.columns:
            renombres["NOMBRE_PRODUCTO"] = "NOMBRE_REPUESTO"
        df = df.rename(columns=renombres)

        requeridas = {
            "CODIGO_REPUESTO",
            "NOMBRE_REPUESTO",
            "CLASIFICACION I",
            "CLASIFICACION II",
            "CLASIFICACION III",
            "CLASIFICACION IV",
        }
        if requeridas.issubset(df.columns):
            df = df[list(requeridas)].copy()
            df["CODIGO_REPUESTO"] = normalizar_codigo(df["CODIGO_REPUESTO"])
            return df.drop_duplicates("CODIGO_REPUESTO", keep="first")

    raise ValueError(
        f"Ninguna hoja de '{ruta.name}' contiene la estructura de tabla maestra esperada."
    )


df_maestro = cargar_tabla_maestra(ruta_tabla_maestra)
print(f"Repuestos en la tabla maestra: {len(df_maestro):,}")
display(df_maestro.head())


## 4. Limpieza de los archivos históricos

Cada columna mensual se transforma en registros de período y cantidad. Se conservan movimientos positivos; los valores vacíos, totales y filas sin código válido se excluyen.


In [ ]:
def cargar_movimientos(ruta):
    hoja, fila_encabezado = detectar_estructura_movimientos(ruta)
    df = pd.read_excel(ruta, sheet_name=hoja, header=fila_encabezado)
    df.columns = [normalizar_texto(columna) for columna in df.columns]

    periodos = [columna for columna in df.columns if es_periodo(columna)]
    if not periodos:
        raise ValueError(f"No se encontraron columnas mensuales en '{ruta.name}'.")

    # La primera columna contiene normalmente '[código] descripción'.
    columnas_no_periodo = [columna for columna in df.columns if columna not in periodos]
    columna_detalle = next(
        (
            columna
            for columna in columnas_no_periodo
            if df[columna].astype("string").str.contains(
                r"\[[^\]]+\]", regex=True, na=False
            ).any()
        ),
        None,
    )
    if columna_detalle is None:
        raise ValueError(f"No se encontró la columna de repuesto en '{ruta.name}'.")

    detalle = (
        df[columna_detalle]
        .astype("string")
        .str.upper()
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )
    df["CODIGO_REPUESTO"] = normalizar_codigo(
        detalle.str.extract(r"\[([^\]]+)\]", expand=False)
    )
    df["REPUESTO_ORIGEN"] = detalle
    df = df[df["CODIGO_REPUESTO"].notna()].copy()

    for periodo in periodos:
        df[periodo] = pd.to_numeric(df[periodo], errors="coerce")

    largo = df.melt(
        id_vars=["CODIGO_REPUESTO", "REPUESTO_ORIGEN"],
        value_vars=periodos,
        var_name="PERIODO",
        value_name="CANTIDAD_MOVIMIENTO",
    )
    largo = largo.dropna(subset=["CANTIDAD_MOVIMIENTO"])
    largo = largo[largo["CANTIDAD_MOVIMIENTO"] > 0].copy()
    largo["PERIODO"] = largo["PERIODO"].map(normalizar_texto)
    largo["ARCHIVO_ORIGEN"] = ruta.name

    return largo, {
        "ARCHIVO": ruta.name,
        "HOJA": hoja,
        "FILA_ENCABEZADO": fila_encabezado,
        "PERIODOS": len(periodos),
        "REGISTROS_VALIDOS": len(largo),
    }


tablas_movimientos = []
resumen_cargas = []

for archivo in archivos_origen:
    tabla, resumen = cargar_movimientos(archivo)
    tablas_movimientos.append(tabla)
    resumen_cargas.append(resumen)

df_movimientos = pd.concat(tablas_movimientos, ignore_index=True)
df_resumen_cargas = pd.DataFrame(resumen_cargas)

if df_movimientos.empty:
    raise ValueError("Los archivos origen no produjeron movimientos positivos válidos.")

display(df_resumen_cargas)
print(f"Movimientos válidos cargados: {len(df_movimientos):,}")


## 5. Clasificación y controles de calidad


In [ ]:
df_mantenimiento = df_movimientos.merge(
    df_maestro,
    on="CODIGO_REPUESTO",
    how="left",
    validate="many_to_one",
)

df_no_clasificados = (
    df_mantenimiento[df_mantenimiento["NOMBRE_REPUESTO"].isna()][
        ["CODIGO_REPUESTO", "REPUESTO_ORIGEN", "ARCHIVO_ORIGEN"]
    ]
    .drop_duplicates()
    .sort_values(["CODIGO_REPUESTO", "ARCHIVO_ORIGEN"])
    .reset_index(drop=True)
)

with pd.ExcelWriter(ruta_no_clasificados, engine="openpyxl") as writer:
    df_no_clasificados.to_excel(
        writer,
        sheet_name="Repuestos_No_Clasificados",
        index=False,
    )

df_mantenimiento = df_mantenimiento[
    df_mantenimiento["NOMBRE_REPUESTO"].notna()
].copy()

columnas_finales = [
    "CODIGO_REPUESTO",
    "NOMBRE_REPUESTO",
    "CLASIFICACION I",
    "CLASIFICACION II",
    "CLASIFICACION III",
    "CLASIFICACION IV",
    "PERIODO",
    "CANTIDAD_MOVIMIENTO",
    "ARCHIVO_ORIGEN",
]
df_mantenimiento = (
    df_mantenimiento[columnas_finales]
    .sort_values(["PERIODO", "CLASIFICACION I", "NOMBRE_REPUESTO"])
    .reset_index(drop=True)
)

if df_mantenimiento.empty:
    raise ValueError("No quedaron movimientos clasificados para generar informes.")
if df_mantenimiento[["CODIGO_REPUESTO", "NOMBRE_REPUESTO", "PERIODO"]].isna().any().any():
    raise ValueError("La tabla final contiene nulos en campos obligatorios.")

print(f"Movimientos clasificados: {len(df_mantenimiento):,}")
print(f"Repuestos no clasificados: {len(df_no_clasificados):,}")
display(df_no_clasificados.head(20))


## 6. Informes mensuales de mantenimiento portuario


In [ ]:
resumen_archivos = []

for periodo, df_periodo in df_mantenimiento.groupby("PERIODO", sort=True):
    periodo_archivo = limpiar_nombre_archivo(periodo)
    ruta_periodo = (
        CARPETA_SALIDA_MENSUAL
        / f"{PREFIJO_MENSUAL}_{periodo_archivo}.xlsx"
    )
    resumen_periodo = pd.DataFrame(
        {
            "INDICADOR_MANTENIMIENTO": [
                "Movimientos registrados",
                "Repuestos diferentes",
                "Cantidad total",
            ],
            "VALOR": [
                len(df_periodo),
                df_periodo["CODIGO_REPUESTO"].nunique(),
                df_periodo["CANTIDAD_MOVIMIENTO"].sum(),
            ],
        }
    )

    with pd.ExcelWriter(ruta_periodo, engine="openpyxl") as writer:
        df_periodo.to_excel(writer, sheet_name="Movimientos_Repuestos", index=False)
        resumen_periodo.to_excel(writer, sheet_name="Resumen_Mantenimiento", index=False)

    resumen_archivos.append(
        {
            "PERIODO": periodo,
            "REGISTROS": len(df_periodo),
            "ARCHIVO_GENERADO": ruta_periodo.name,
        }
    )

df_resumen_archivos = pd.DataFrame(resumen_archivos)
print(f"Informes mensuales generados en: {CARPETA_SALIDA_MENSUAL}")
display(df_resumen_archivos)


## 7. Informe consolidado para Power BI


In [ ]:
resumen_periodos = (
    df_mantenimiento.groupby("PERIODO", as_index=False)
    .agg(
        MOVIMIENTOS=("CODIGO_REPUESTO", "size"),
        REPUESTOS_DIFERENTES=("CODIGO_REPUESTO", "nunique"),
        CANTIDAD_TOTAL=("CANTIDAD_MOVIMIENTO", "sum"),
    )
    .sort_values("PERIODO")
)

with pd.ExcelWriter(ruta_salida_powerbi, engine="openpyxl") as writer:
    df_mantenimiento.to_excel(
        writer,
        sheet_name="Movimientos_Repuestos",
        index=False,
    )
    resumen_periodos.to_excel(
        writer,
        sheet_name="Resumen_Mantenimiento",
        index=False,
    )
    df_resumen_cargas.to_excel(
        writer,
        sheet_name="Control_Cargas",
        index=False,
    )

print(f"Informe consolidado generado: {ruta_salida_powerbi}")
print(f"Total de registros: {len(df_mantenimiento):,}")
display(resumen_periodos)


## Resultado

El proceso genera:

- `Informe_Consolidado_Movimientos_Repuestos_Mantenimiento_Portuario.xlsx`: consolidado para Power BI.
- `Informe_Movimientos_Repuestos_Mantenimiento_Portuario_<PERIODO>.xlsx`: informes mensuales.
- `Informe_Repuestos_No_Clasificados_Mantenimiento_Portuario.xlsx`: catálogo auxiliar para revisión.

Todos los nombres y hojas están relacionados con la gestión de mantenimiento de equipos portuarios.
